## Lab 5: Clean the bike rentals ## 

Today we cleaned the Airbnb data step by step: empty columns, the price from text to numbers, t
and f, dates, missing ratings and the bathrooms text. In this lab you clean new data: the rentals of
a bike shop in Stockholm. The shop and its data are made up, but the problems are the kinds
that real data has. This lab has two parts, and you choose.

Part 1: Practise repeats the steps from today on the new table. The problems are of the same
kinds as today, in new columns. Do this part if you want to try the whole routine once more on your
own.

Part 2: Go deeper is for you who felt "I understood this, and I want to go further". The questions
are more open, and you have to decide what is right when the data does not give a clear answer.
I do not expect you to do both parts. It is your own choice, and you are welcome to contact me if
you want to talk about which part to take.
Download bike_rentals.csv from Teams and save it in your data folder. Work in a new notebook.
Keep the original: load the file into rentals_raw and work on a copy. Write a short comment above
each step, in your own words. If you get stuck: read the error message from the bottom up, print
the thing you are working with, and look at the lecture notebook. Then ask.

## Part 1: Practise ## 

Clean the new table with the same steps as today.

Tools you need

rentals_raw = pd.read_csv('data/bike_rentals.csv')
rentals = rentals_raw.copy()
rentals.columns[rentals.isna().all()] # names of the empty columns
rentals.dropna(axis=1, how='all') # remove the empty columns
rentals['x'].str.replace('abc', '', regex=False)
rentals['x'].str.strip().str.lower().str.title()
rentals['x'].astype(float)
rentals['x'].map({'t': True, 'f': False})
[c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
pd.to_datetime(rentals['x'], errors='coerce')
(rentals['end'] - rentals['start']).dt.days
pd.crosstab(a, b)
text.str.contains('word')
text.str.extract(r'(\d+)', expand=False)
number.where(condition, value) # keeps the number where True, else the value
rentals.duplicated().sum()
rentals.drop_duplicates().reset_index(drop=True)

## Tasks - 1  ##

** 1.1 Load it, and remove the empty columns **

Load bike_rentals.csv and make a copy. 
Print the shape and the data types. 
Which type does price have? 
Then find the columns where every value is missing, print their names,and remove them. 
How many columns are left, and how many empty columns?

In [ ]:
# 1.1 Load it, and remove the empty columns

import pandas as pd

# Load bike_rentals.csv and make a copy
rentals_raw = pd.read_csv('data/bike_rentals.csv')
rentals = rentals_raw.copy()

# Print shape and data types
print("--- Shape of rentals dataset ---")
print(rentals.shape)

print("\n--- Data types ---")
print(rentals.dtypes)

# Check the data type of 'price' specifically
print(f"\nData type of 'price' column: {rentals['price'].dtype}")

# Find columns where every value is missing
empty_cols = rentals.columns[rentals.isna().all()].tolist()
print(f"\nEmpty columns found (all values missing): {empty_cols}")
print(f"Total number of empty columns: {len(empty_cols)}")

# Remove the empty columns (axis=1, how='all')
rentals = rentals.dropna(axis=1, how='all')

# Check remaining dimensions
print(f"\nShape after dropping empty columns: {rentals.shape}")
print(f"Number of columns left: {rentals.shape[1]}")

--- Shape of rentals dataset ---
(403, 15)

--- Data types ---
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object

Data type of 'price' column: str

Empty columns found (all values missing): ['coupon_code', 'damage_notes', 'late_fee_note']
Total number of empty columns: 3

Shape after dropping empty columns: (403, 12)
Number of columns left: 12


** 1.2 The price: from text to numbers **

Look at the first rows of price. It is text with a currency in front and a comma for
thousands. 
Remove the two things in two separate steps, and print the first three values
after each step. 
Then change the type to float, check the type, and run describe() on the
price. 
Finally, show the three most expensive rentals (rental_id, bike_type, status andprice).

In [4]:
# 1.2. The price: from text to numbers

# Check the first few values of price before cleaning
print("--- First 3 values (Original) ---")
print(rentals['price'].head(3))

# Remove the currency symbol/text (e.g., 'kr') and strip whitespace:
rentals['price'] = rentals['price'].str.replace('kr', '', regex=False).str.strip()
print("\n--- First 3 values after removing currency text ---")
print(rentals['price'].head(3))

# Remove the currency symbol ('SEK') and strip whitespace:
rentals['price'] = rentals['price'].str.replace('SEK', '', regex=False).str.strip()
print("\n--- First 3 values after removing currency text ---")
print(rentals['price'].head(3))

# Remove separator (commas):
rentals['price'] = rentals['price'].str.replace(',', '', regex=False)
print("\n--- First 3 values after removing commas ---")
print(rentals['price'].head(3))

# Change the data type to float and verify
rentals['price'] = rentals['price'].astype(float)
print(f"\nNew data type of 'price': {rentals['price'].dtype}")

# Run describe() on the price column
print("\n--- Summary Statistics for Price ---")
print(rentals['price'].describe())

# show the three most expensive rentals with specific columns
cols_to_display = ['rental_id', 'bike_type', 'status', 'price']
top_expensive = rentals.nlargest(3, 'price')[cols_to_display]

print("\n--- Three Most Expensive Rentals ---")
print(top_expensive)

--- First 3 values (Original) ---
0     SEK 855.50
1    SEK 1311.50
2     SEK 350.00
Name: price, dtype: str

--- First 3 values after removing currency text ---
0     SEK 855.50
1    SEK 1311.50
2     SEK 350.00
Name: price, dtype: str

--- First 3 values after removing currency text ---
0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

--- First 3 values after removing commas ---
0     855.50
1    1311.50
2     350.00
Name: price, dtype: str

New data type of 'price': float64

--- Summary Statistics for Price ---
count     352.000000
mean      847.348011
std       633.435513
min       154.500000
25%       406.000000
50%       668.250000
75%      1194.625000
max      3401.000000
Name: price, dtype: float64

--- Three Most Expensive Rentals ---
    rental_id   bike_type     status   price
187    R-5282  Cargo bike  Completed  3401.0
49     R-5209  Cargo bike  Completed  2979.5
270    R-5378      E-bike  Completed  2941.0


** 1.3 The missing prices **

How many prices are missing, and how many percent is that? 
Is there a reason? Compare with the column status: use pd.crosstab on "price is missing" and "the rental was
cancelled". 
Which missing prices have a reason, and which are missing by accident? 
Write what you decide to do with them, as a comment.

In [ ]:
# 1.3. The missing prices

# Count missing prices and calculate the percentage
missing_prices_count = rentals['price'].isna().sum()
total_rentals = len(rentals)
missing_prices_pct = (missing_prices_count / total_rentals) * 100

print(f"Number of missing prices: {missing_prices_count}")
print(f"Percentage of missing prices: {missing_prices_pct:.2f}%")

# Create a boolean mask for missing prices and cross-tabulate with 'status'
price_is_missing = rentals['price'].isna()

print("\n--- Crosstab: Price is Missing vs. Rental Status ---")
print(pd.crosstab(price_is_missing, rentals['status'], margins=True))

# --- (Comment) - Which have a reason vs. by accident:
#   If missing prices align exclusively with cancelled rentals (status == 'Cancelled'), 
#   they are missing because no financial transaction occurred - the booking was cancelled.
#   If prices are missing on completed or active rentals, they are missing by accident - 
#     it can be system/logging error.
#
# - What we decide to do:
#   We can keep missing prices as NaN (or explicitly fill with 0) for cancelled rentals where no charge applies.
#   For any unexpected missing price records on successful/completed rentalsbut we cam mark it or drop it.

Number of missing prices: 51
Percentage of missing prices: 12.66%

--- Crosstab: Price is Missing vs. Rental Status ---
status  Cancelled  Completed  No-show  All
price                                     
False           0        330       22  352
True           47          4        0   51
All            47        334       22  403


** 1.4 The letters t and f **

Find the columns that contain only t and f (and maybe missing values), with a list
comprehension. Why is status not in the list? 
Change the columns to True and False with map. 
How many missing values does each column have afterwards? 
Then compare the median price of members and non-members, with the number of rentals behind each number.

In [6]:
# 1.4. The letters t and f

# Find columns that contain only 't' and 'f' (and maybe missing values) using a list comprehension
t_f_cols = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
print(f"Columns with only 't' and 'f': {t_f_cols}")

# Change these columns to True and False using map
for col in t_f_cols:
    rentals[col] = rentals[col].map({'t': True, 'f': False})

# Check how many missing values each column has afterwards
print("\n--- Missing values in t/f columns after conversion ---")
print(rentals[t_f_cols].isna().sum())

# Compare the median price of members and non-members, with rental counts
member_col = 'member' if 'member' in rentals.columns else [c for c in rentals.columns if 'member' in c.lower()][0]

price_comparison = rentals.groupby(member_col, observed=False)['price'].agg(
    median_price='median',
    rental_count='count'
)

print(f"\n--- Median Price & Rental Counts by Membership ({member_col}) ---")
print(price_comparison)

Columns with only 't' and 'f': ['is_member', 'insured', 'paid_online']

--- Missing values in t/f columns after conversion ---
is_member      0
insured        3
paid_online    0
dtype: int64

--- Median Price & Rental Counts by Membership (is_member) ---
           median_price  rental_count
is_member                            
False            756.25           248
True             558.00           104


** 1.5 The dates **

start_date and end_date are text. 
Count the missing values in both columns before,
change them to dates with errors='coerce', and count again. Did something change?

Check the types. Then create the column rental_days (the end minus the start, in days)
and run describe() on it. 
Something looks wrong at the minimum. 
Write what you see as a comment. You do not need to fix it in this part.

In [7]:
# 1.5. The dates

# Automatically detect whether columns are named 'start_date'/'end_date' or 'start'/'end':
start_col = 'start_date' if 'start_date' in rentals.columns else 'start'
end_col = 'end_date' if 'end_date' in rentals.columns else 'end'

# Count missing values in both date columns before conversion:
print("--- Missing values before conversion ---")
print(rentals[[start_col, end_col]].isna().sum())

# Change them to dates using pd.to_datetime with errors='coerce'
rentals[start_col] = pd.to_datetime(rentals[start_col], errors='coerce')
rentals[end_col] = pd.to_datetime(rentals[end_col], errors='coerce')

# Count missing values after conversion and check data types:
print("\n--- Missing values after conversion (errors='coerce') ---")
print(rentals[[start_col, end_col]].isna().sum())

print("\n--- Data types ---")
print(rentals[[start_col, end_col]].dtypes)

# Create the column 'rental_days' (end minus start, in days) and run describe()
rentals['rental_days'] = (rentals[end_col] - rentals[start_col]).dt.days

print("\n--- Summary Statistics for rental_days ---")
print(rentals['rental_days'].describe())

# --- Something looks wrong at the minimum (Comment) ---
# The minimum value for 'rental_days' is negative (e.g., -1 or lower). 
# This indicates bad data or logging errors where the recorded end date/time 
# occurs "before" the start date/time.

--- Missing values before conversion ---
start_date    0
end_date      0
dtype: int64

--- Missing values after conversion (errors='coerce') ---
start_date    2
end_date      0
dtype: int64

--- Data types ---
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object

--- Summary Statistics for rental_days ---
count    401.000000
mean       2.955112
std        1.835206
min       -2.000000
25%        1.000000
50%        3.000000
75%        4.000000
max        7.000000
Name: rental_days, dtype: float64


** 1.6 The missing ratings **

How many ratings are missing? 
Compare with status with pd.crosstab. 
In the lecture the missing ratings were exactly the listings without reviews. 
Is it a perfect match this time?
Create a column has_rating and count it. 
Run describe() on rating and write down the median.

In [8]:
# 1.6. The missing ratings

# Count how many ratings are missing:
missing_ratings_count = rentals['rating'].isna().sum()
total_rentals = len(rentals)
missing_ratings_pct = (missing_ratings_count / total_rentals) * 100

print(f"Number of missing ratings: {missing_ratings_count} ({missing_ratings_pct:.2f}% of all rentals)")

# Compare missing ratings with 'status' using pd.crosstab
rating_is_missing = rentals['rating'].isna()

print("\n--- Crosstab: Rating is Missing vs. Rental Status ---")
print(pd.crosstab(rating_is_missing, rentals['status'], margins=True))

# Create a column 'has_rating' and count it
rentals['has_rating'] = rentals['rating'].notna()

print("\n--- Count of has_rating ---")
print(rentals['has_rating'].value_counts())

# Run describe() on rating and inspect the median
print("\n--- Summary Statistics for Rating ---")
print(rentals['rating'].describe())

rating_median = rentals['rating'].median()
print(f"\nExact Median Rating: {rating_median}")

Number of missing ratings: 115 (28.54% of all rentals)

--- Crosstab: Rating is Missing vs. Rental Status ---
status  Cancelled  Completed  No-show  All
rating                                    
False           0        288        0  288
True           47         46       22  115
All            47        334       22  403

--- Count of has_rating ---
has_rating
True     288
False    115
Name: count, dtype: int64

--- Summary Statistics for Rating ---
count    288.000000
mean       4.225694
std        1.019547
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

Exact Median Rating: 5.0


** 1.7 The helmets text **
Look at helmets_text with value_counts(dropna=False). 
Create two new columns: child_helmet 
(True if the text contains the word child) and helmets_count (the number in the text). 
The text "No helmets" has no number. 
What does the number get for those rows, and how do you make it 0 instead? 
Count the values in helmets_count.

In [12]:
# 1.7. The helmets text

# Check - helmets_text value counts including missing values:
print("--- Value counts of helmets_text ---")
print(rentals['helmets_text'].value_counts(dropna=False))

# Create child_helmet column (True if the text contains the word 'child')
rentals['child_helmet'] = rentals['helmets_text'].str.contains('child', case=False, na=False)

# Count the values in helmets_count
print("\n--- Value counts of helmets_count ---")
print(rentals['helmets_count'].value_counts(dropna=False))

--- Value counts of helmets_text ---
helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64

--- Value counts of helmets_count ---
helmets_count
1    193
2    127
0     63
3     20
Name: count, dtype: int64


** 1. 8 Spellings and duplicates **

For station, bike_type and status: how many different values are there, 
and how many if you take away spaces and capital letters (strip and lower)? 
Which column has a problem?
Fix it with strip and title, and count the rentals per station. 
Then check for complete duplicate rows and for duplicate rental_id. 
Remove the duplicates and print the final shape. 

In [13]:
# 1.8. Spellings and duplicates

# Check unique values before and after strip/lower for station, bike_type, and status
cols_to_check = ['station', 'bike_type', 'status']

print("--- Unique Values Comparison ---")
for col in cols_to_check:
    if col in rentals.columns:
        orig_unique = rentals[col].nunique(dropna=False)
        clean_unique = rentals[col].astype(str).str.strip().str.lower().nunique()
        print(f"Column '{col}': Original unique = {orig_unique}, Cleaned unique = {clean_unique}")

# Fix the inconsistent column (typically 'station') with strip and title
if 'station' in rentals.columns:
    rentals['station'] = rentals['station'].str.strip().str.title()

# Count the rentals per station
print("\n--- Rentals per Station ---")
print(rentals['station'].value_counts())

# Check for complete duplicate rows and duplicate rental_ids
complete_dups = rentals.duplicated().sum()
print(f"\nComplete duplicate rows: {complete_dups}")

if 'rental_id' in rentals.columns:
    id_dups = rentals.duplicated(subset=['rental_id']).sum()
    print(f"Duplicate rental_id count: {id_dups}")

# Remove duplicates and print the final shape
print(f"\nInitial shape before removing duplicates: {rentals.shape}")

rentals = rentals.drop_duplicates().reset_index(drop=True)
if 'rental_id' in rentals.columns:
    rentals = rentals.drop_duplicates(subset=['rental_id']).reset_index(drop=True)

print(f"Final shape after cleaning and removing duplicates: {rentals.shape}")

--- Unique Values Comparison ---
Column 'station': Original unique = 20, Cleaned unique = 5
Column 'bike_type': Original unique = 3, Cleaned unique = 3
Column 'status': Original unique = 3, Cleaned unique = 3

--- Rentals per Station ---
station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64

Complete duplicate rows: 3
Duplicate rental_id count: 3

Initial shape before removing duplicates: (403, 16)
Final shape after cleaning and removing duplicates: (400, 16)


## Expected ## 

Task 1: (403, 15). price is text (str). The empty columns: coupon_code, damage_notes and
late_fee_note. After removing them: (403, 12) and 0 empty columns.

Task 2: after step 1 ['855.50', '1,311.50', '350.00'], after step 2 ['855.50', '1311.50',
'350.00']. The type is float64. describe(): count 352, mean 847.3, std 633.4, min 154.5, median
668.2, max 3401.0. The top three: R-5282 (Cargo bike, 3401.0), R-5209 (Cargo bike, 2979.5), R-5378
(E-bike, 2941.0).

Task 3: 51 missing (12.7 percent). 47 of them are cancelled rentals (missing for a reason: nothing
was paid). 4 are not cancelled (missing by accident). Good answer: leave them as missing, and write
how many rentals are behind each number.

Task 4: ['is_member', 'insured', 'paid_online']. status has three different texts, not only t
and f. Missing: insured 3, the others 0. is_member: False 288 and True 115. Median price:
non-members 756 (248 rentals), members 558 (104 rentals). The members pay less, but do not say
yet that membership is the reason.

Task 5: before: 0 missing in both. After: start_date 2 missing (two impossible dates, for example 30
February), end_date 0. The types are datetime64[us]. rental_days: count 401, mean 2.96, min -2,
max 7. Two rentals end before they start (R-5077 and R-5217).

Task 6: 115 missing. 47 cancelled, 22 no-shows and 46 completed: not a perfect match. A
completed rental can be without a rating because the customer did not give one. has_rating: True
288, False 115. Median rating 5.0 (mean 4.23).

Task 7: 1 helmet 146, 2 helmets 127, No helmets 59, 1 child helmet 47, 3 helmets 20, missing 4.
"No helmets" has no number, so the extraction gives NaN. With where(~text.str.contains('no
helmets'), 0) it becomes 0. helmets_count: 0 → 59, 1 → 193, 2 → 127, 3 → 20, missing 4.
child_helmet: 47 True.

Task 8: station 20 different values, and 5 after strip and lower. bike_type 3 and 3, status 3 and
3. After the fix: Central 107, Södermalm 95, Kungsholmen 89, Östermalm 57, Hammarby 55. 3
duplicate rows and 3 duplicate rental_id. Final shape: (400, 16).

## Part 2: Go deeper ##

Here you ask what the problems mean, you put all the cleaning in one cell and you document your
decisions. The tools are in the box below.

** Tools you need **

rentals.groupby('station').agg(n=('price', 'count'), median=('price', 'median'))
(rentals['bike_type'] == 'E-bike').groupby(rentals['station']).mean() * 100
rentals[rentals['rental_days'] < 0]
rentals['rental_days'].where(rentals['rental_days'] >= 0) # else: missing
rentals_clean.to_csv('data/bike_rentals_clean.csv', index=False)
pd.read_csv('data/bike_rentals_clean.csv', parse_dates=['start_date', 'end_date'])

## Tasks - 2  ##

**. Part 2.1 Which station is the most expensive? **

Start from your clean table (400 rows). 
(a) Calculate the number of rentals with a price and the median price for every station, highest median first. (b) Do the same, but only for the e-bikes. 
(c) Calculate the share of e-bikes at each station, with the mean of True and False times 100. 
Does the order change between (a) and (b)? 
How many rentals are behind each number? 
Write one sentence that you can say, and one that you cannot.

In [2]:
#**. Part 2.1 Which station is the most expensive? **

# (a) All rentals: count of non-missing prices and median price per station, highest median first

station_all = (
    rentals_clean.groupby('station', dropna=False)
    .agg(
        n=('price', 'count'),
        median=('price', 'median')
    )
    .sort_values('median', ascending=False)
)

print("--- (a) All rentals by station (highest median first) ---")
print(station_all)

# (b) E-bikes only: count of non-missing prices and median price per station, highest median first
ebike_only = rentals_clean[rentals_clean['bike_type'] == 'E-bike']
station_ebike = (
    ebike_only.groupby('station', dropna=False)
    .agg(
        n=('price', 'count'),
        median=('price', 'median')
    )
    .sort_values('median', ascending=False)
)

print("\n--- (b) E-bikes only by station (highest median first) ---")
print(station_ebike)

# (c) Share of e-bikes at each station (mean of True/False * 100)
ebike_share = (
    (rentals_clean['bike_type'] == 'E-bike')
    .groupby(rentals_clean['station'], dropna=False)
    .mean() * 100
)

summary = station_all.join(
    station_ebike.rename(columns={'n': 'ebike_n', 'median': 'ebike_median'})
).join(ebike_share.rename('ebike_pct'))

print("\n--- Combined view: all rentals, e-bike median, sample sizes (n), and e-bike share (%) ---")
print(summary.round({'median': 2, 'ebike_median': 2, 'ebike_pct': 1}))

NameError: name 'rentals_clean' is not defined

** 2.2 Values that cannot be true **

Show the rentals where rental_days is below 0. 
A rental cannot end before it starts. 
Use where to turn these values into missing values. 
How many rental_days are missing now, and why is it more than two? 
What happens to the mean? Write what you would ask the shop, if you could.

** 2.3 Everything in one cell, and save **

Write one code cell that starts from rentals_raw and does all the steps: 
remove the empty columns, change the price, t and f, the dates, rental_days 
(with the impossible values set to missing), has_rating, child_helmet, 
helmets_count, the spelling of station and the duplicates. 

Save the result as data/bike_rentals_clean.csv with index=False. 
Then read the file back with parse_dates and print the types of price, is_member, insured and
start_date. Which one is not what you expected, and why?

** 2.4 Write down your decisions **

In a markdown cell, make a list of every problem you found in the table: 
empty columns, the price as text, missing prices, impossible dates, 
rentals that end before they start, missing ratings, the helmets text, 
the spellings of station and the duplicates. 
For each, write what you did (remove, fill, change or leave), 
why, and what it does to the numbers. 
A colleague should be able to read your list and know exactly what you did.

** 2.5 Your own question **

Come up with one question about the bike shop that your clean table can answer. 
Write the question as a comment, answer it with code, and finish with a comment 
about what the answer does not tell you. 
That last part is the most important one.

## Expected ## 

Task 1: (a) Kungsholmen 73 rentals, median 742; Södermalm 87, 720; Östermalm 48, 710;
Hammarby 47, 609; Central 95, 596. (b) E-bikes: Kungsholmen 22, 1554; Södermalm 25, 1220;
Central 27, 1144; Östermalm 19, 1088; Hammarby 15, 712. (c) Share of e-bikes: Östermalm 39,
Hammarby 31, Södermalm 31, Kungsholmen 30, Central 28 (percent). Kungsholmen is first in both.
Central goes from last to third. The e-bike share is about the same everywhere, so the mix of bikes
does not explain the order. A fair sentence: "Kungsholmen has the highest median, also for e-bikes".
Not a fair sentence: "Kungsholmen is the most expensive station", because only 15 to 27 e-bike
rentals stand behind each number.

Task 2: 2 rentals: R-5077 (-1 day) and R-5217 (-2 days). After where: 4 missing (the 2 impossible
start dates and the 2 negative values). The mean goes from 2.96 to 2.98. A good question to the
shop: were the dates typed in the wrong order?

Task 3: shape (400, 16) before saving, and (400, 16) after reading it back. Types after reading:
price float64, is_member bool, insured object (it has 3 missing values, and a CSV file cannot keep
True, False and missing as a bool type), start_date datetime64[us] (only because of parse_dates).

Task 4: open. A good list has a decision, a reason and an effect for each problem. For example:
empty columns (3) removed, nothing is lost; price (text) changed to a number; missing prices (51)
left as missing, 47 are cancelled rentals; impossible dates (2) turned into missing; negative days (2)
turned into missing; missing ratings (115) left, 46 are completed rentals; helmets text turned into a
number, "No helmets" is 0; spellings of station (20 to 5) fixed with strip and title; duplicates (3)
removed.